In [2]:
import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm

# Data Preprocessing

In [3]:
#load the data
pokemon_bio = pd.read_csv('./pokemon.csv')
combats = pd.read_csv('./combats.csv')

In [3]:
#split the data to training set and test set using 70%/30% probability
split = 0.7
cut = round(combats.shape[0] * split)
shuffle_ls = np.random.choice(combats.shape[0], size=combats.shape[0], replace=False)

train_ind = shuffle_ls[:cut]
test_ind = shuffle_ls[cut:]

train_combats = combats.iloc[train_ind, :]
test_combats = combats.iloc[test_ind, :]

In [4]:
#np.save('train_ind_.npy', train_ind)

In [4]:
#this is the map between pre and post mega evolved pokemons id, can be achieved from pokemon.csv
evmap = {4: 3, 8: 7, 9: 7, 13: 12, 20: 19, 24: 23, 72: 71, 88: 87, 103: 102, 125: 124, 138: 137,
        142: 141, 155: 154, 164: 163, 165: 163, 197: 196, 225: 224, 230: 229, 233: 232, 249: 248, 
        269: 268, 276: 275, 280: 279, 284: 283, 307: 306, 328: 327, 330: 329, 334: 333, 337: 336, 
        340: 339, 350: 349, 355: 354, 367: 366, 388: 387, 394: 393, 398: 397, 410: 409, 414: 413, 
        419: 418, 421: 420, 427: 426, 477: 476, 495: 494, 499: 498, 512: 511, 528: 527, 592: 591, 797: 796}

cut = 28
shuffle_ls = np.random.choice(len(evmap), size=len(evmap), replace=False)
test_ind = shuffle_ls[:cut]

mega = list(evmap.keys())

indexlist_test = []
indexdict_test = {}
for i in range(cut):
    indexlist_test.append(mega[test_ind[i]])
    indexdict_test[indexlist_test[i]] = i

In [14]:
# if any of these ids is included in the indexlist_test, the graph won't be connected.
# as a result, we generate the shuffle_ls (in the previous cell) for several times to make sure none
# of the following ids is included
13, (8, 9), (164, 165), 419, 398, 334

(13, (8, 9), (164, 165), 419, 398, 334)

In [11]:
# check if any of these ids is included
indexlist_test
print(13 in indexlist_test)
print(8 in indexlist_test and 9 in indexlist_test)
print(164 in indexlist_test and 165 in indexlist_test)
print(419 in indexlist_test and 398 in indexlist_test)
print(334 in indexlist_test)

False
False
False
True
True


In [38]:
#np.save('indexlist_test_.npy', indexlist_test)

In [60]:

indexlist_train_init = []
for i in range(1, 801):
    if i not in indexlist_test:
        indexlist_train_init.append(i)
        
indexdict_train_init = {}
for i in range(len(indexlist_train_init)):
    indexdict_train_init[indexlist_train_init[i]] = i

In [61]:
# this function generate the trading data, which includes the graph and covariates. The covariates are normalized here
def generate_train_data(train_combats, pokemon_bio, indexlist_train, indexdict_train):
    n = len(indexlist_train)
    indexset_train = set(indexlist_train)
    X = np.zeros([n, 3])
    for i in range(len(indexlist_train)):
        X[i, 0] = np.log(pokemon_bio.iloc[indexlist_train[i] - 1]["HP"])
        X[i, 1] = np.log(pokemon_bio.iloc[indexlist_train[i] - 1]["Attack"])
        #X[i, 2] = np.log(pokemon_bio.iloc[indexlist_train[i] - 1]["Defense"])
        X[i, 2] = int(indexlist_train[i] in mega)
        
    means = np.zeros(3)
    stds = np.zeros(3)
    means[0] = np.mean(X[:, 0])
    stds[0] = np.std(X[:, 0], ddof = 1)
    means[1] = np.mean(X[:, 1])
    stds[1] = np.std(X[:, 1], ddof = 1)
    means[2] = np.mean(X[:, 2])
    stds[2] = np.std(X[:, 2], ddof = 1)
    
    for j in range(3):
        X[:, j] = (2/27) * (X[:, j] - means[j]) / stds[j]
    
    y = np.zeros([n, n]) 
    graph = np.zeros([n, n])
    for i in range(len(train_combats)):
        if train_combats.iloc[i]['First_pokemon'] in indexset_train and train_combats.iloc[i]['Second_pokemon'] in indexset_train:
            graph[indexdict_train[train_combats.iloc[i]['First_pokemon']], indexdict_train[train_combats.iloc[i]['Second_pokemon']]] += 1
            graph[indexdict_train[train_combats.iloc[i]['Second_pokemon']], indexdict_train[train_combats.iloc[i]['First_pokemon']]] += 1
            if train_combats.iloc[i]['First_pokemon'] == train_combats.iloc[i]['Winner']:
                y[indexdict_train[train_combats.iloc[i]['Second_pokemon']], indexdict_train[train_combats.iloc[i]['First_pokemon']]] += 1
            else:
                y[indexdict_train[train_combats.iloc[i]['First_pokemon']], indexdict_train[train_combats.iloc[i]['Second_pokemon']]] += 1
    
    return X, y, graph, means, stds

In [62]:
_, _, graph_init, _, _ = generate_train_data(train_combats, pokemon_bio, indexlist_train_init, indexdict_train_init)
_, labels = connected_components(csgraph=graph_init, directed=False, return_labels=True)

In [63]:
indexlist_train = []
for i in range(len(indexlist_train_init)):
    if labels[i] == 0:
        indexlist_train.append(indexlist_train_init[i])
        
indexdict_train = {}
for i in range(len(indexlist_train)):
    indexdict_train[indexlist_train[i]] = i

In [64]:
X, y, graph, means, stds = generate_train_data(train_combats, pokemon_bio, indexlist_train, indexdict_train)

# Sparse Ranking Model

In [65]:
# this is the main class to fit the sparse ranking model
# the model is fitted using gradient descent
# after fitting the model, we conduct goodness-of-fit test and calculate the rank confidence interval
class Ranking:
    def __init__(self, X, y, graph):
        self.n = X.shape[0]
        
        self.d = 3
        self.tX = np.zeros([self.n, self.n + 3])
        self.X = X
        self.tX[:, :self.n] = np.eye(self.n)
        self.tX[:, self.n:] = self.X
        
        self.y = y
        self.graph = graph
        
    def fix_support(self, v):
        support = (v[:self.n] != 0)
        self.supportsize = int(sum(support))
        self.estimatedsupport = support
        count = 0
        tX_compressed = np.zeros([self.n, self.supportsize + self.d])
        tX_supported = np.zeros([self.n, self.n + self.d])
        for i in range(self.n):
            tX_compressed[i, self.supportsize : self.supportsize + self.d] = self.X[i, :]
            tX_supported[i, self.n : self.n + self.d] = self.X[i, :]
            if support[i]:
                tX_compressed[i, count] = 1
                count += 1
                tX_supported[i, i] = 1
                
        self.tX_compressed = tX_compressed
        self.tX_supported = tX_supported
    
    def soft_thres(self, v, thres):
        return np.sign(v) * np.maximum(np.abs(v) - thres, 0)
                    
    def gradient(self, v): 
        score = self.tX @ v
        gradient = np.zeros(self.n + self.d)
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] >= 1:
                    prob = np.exp(score[i]) / (np.exp(score[i]) + np.exp(score[j]))
                    gradient += (prob * self.graph[j,i] - self.y[j,i]) * (self.tX[i,:] - self.tX[j,:])
        return gradient
    
    def gradient_supported(self, v):
        score = self.tX @ v
        gradient = np.zeros(self.n + self.d)
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] >= 1:
                    prob = np.exp(score[i]) / (np.exp(score[i]) + np.exp(score[j]))
                    gradient += (prob * self.graph[j,i] - self.y[j,i]) * (self.tX_supported[i,:] - self.tX_supported[j,:])
        return gradient
    
    def Hessian(self, v):
        score = self.tX @ v
        H = np.zeros([self.n + self.d, self.n + self.d])
        for i in range(self.n):
            for j in range(i):
                if self.graph[i,j] >= 1:
                    coeff = self.graph[j,i] * np.exp(score[i] + score[j]) / ((np.exp(score[i]) + np.exp(score[j])) ** 2)
                    contrast = (self.tX[i,:]-self.tX[j,:])[np.newaxis]
                    H += coeff * contrast.T * contrast
        return H
    
    def compressedH(self, v):
        H = self.Hessian(v)
        H_compressed = np.zeros([self.supportsize + self.d, self.supportsize + self.d])
        counti = 0
        for i in range(self.n):
            if self.estimatedsupport[i]:
                countj = 0
                for j in range(self.n):
                    if self.estimatedsupport[j]:
                        H_compressed[counti, countj] = H[i, j]
                        countj += 1
                H_compressed[counti, self.supportsize:] = H[i, self.n:]
                
                counti += 1
                
        for i in range(self.d):
            countj = 0
            for j in range(self.n):
                if self.estimatedsupport[j]:
                    H_compressed[self.supportsize + i, countj] = H[self.n + i, j]
                    countj += 1
            H_compressed[self.supportsize + i, self.supportsize:] = H[self.n + i, self.n:]

        return H_compressed
    
    def diamondinv(self, v):
        H = self.Hessian(v)
        diamond = np.zeros([self.n + self.d, self.n + self.d])
        for i in range(self.n):
            diamond[i, i] = 1 / H[i, i]
        diamond[self.n:, self.n:] = np.linalg.inv(H[self.n:, self.n:])
        return diamond
    
    def generate_MLE(self, tau = 5e-4, lamb = 5):
        eta = 0.001
        v_prev1 = np.zeros(self.n + self.d)
        v_prev2 = np.zeros(self.n + self.d)
        for i in range(300):
            tmp = v_prev1 + (i / (i + 3)) * (v_prev1 - v_prev2) 
            gradient = self.gradient(tmp)
            v = (1 - eta * tau) * tmp - eta * gradient
            v[:self.n] = np.sign(v[:self.n]) * np.maximum(0, np.abs(v[:self.n]) - eta * lamb)
            if np.linalg.norm(v - v_prev1) < 1e-5:
                return v
            else:
                print(i, np.linalg.norm(v - v_prev1))
                v_prev2 = v_prev1
                v_prev1 = v
        return v
        
    def generate_MLE_support(self):
        eta = 0.001
        v_prev = np.zeros(self.n + self.d)
        for i in range(300):
            gradient = self.gradient_supported(v_prev)
            v = v_prev - eta * gradient
            if np.linalg.norm(v - v_prev) < 1e-5:
                return v
            else:
                print(i, np.linalg.norm(v - v_prev))
                v_prev = v
        return v

    def debias_test(self, v, m = 0):
        g = self.gradient(v)
        H = self.Hessian(v)
        result = v[m] - (g[m] / H[m, m])
        return (result - self.tb[m]) * np.sqrt(H[m, m])
    
    def error(self, v):
        return np.max(np.abs(v[:self.n] - self.alpha)), np.linalg.norm(v[self.n:] - self.beta) / np.linalg.norm(self.beta)
    
    def GMB2(self, v, num, m = 0):
        H_compressed = self.compressedH(v)
        H_inv = np.linalg.inv(H_compressed)   
        
        sigma_recorded = np.zeros(self.n)
        up_all_recorded = np.zeros([self.n, self.n])
        scores_recorded = self.tX @ v
        for k in range(self.n):
            if k != m:
                s = 0
                diff = self.tX_compressed[m, :] - self.tX_compressed[k, :]
                tmp = H_inv @ diff
                sigma_recorded[k] = np.sqrt(diff.T @ H_compressed @ diff)
                up_all_recorded[k, :] = tmp @ self.tX_compressed.T
                if sigma_recorded[k] == 0 :
                    print(diff, tmp)
                    
        results = []
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    tmp = scores_recorded[i] - scores_recorded[j]
                    tmp = np.exp(tmp) / (1 + np.exp(tmp))
                    hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i]) 
                    hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j])
                    if self.graph[i, j] > 0:
                        hada[i, j] /= self.graph[i, j]
            for k in range(self.n):
                if k != m:
                    s = 0
                    for i in range(self.n):
                        for j in range(i):
                            up = up_all_recorded[k, i] - up_all_recorded[k, j]
                            s += up * hada[i, j] / sigma_recorded[k]
                    if np.abs(s) > result:
                        result = np.abs(s)
            results.append(result)
            
        return results
 
    def CI2(self, v, m = 0):
        tmp = self.GMB2(v, 200, m)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        print(tmp, c)
        
        scores_recorded = self.tX @ v
        
        H_compressed = self.compressedH(v)
        H_inv = np.linalg.inv(H_compressed)
        result = np.zeros([self.n - 1, 2])
        count = 0
        T = 0
        for k in range(self.n):
            if k != m:
                diff = self.tX_compressed[m, :] - self.tX_compressed[k, :]
                sigma = np.sqrt(diff.T @ H_inv @ diff)
                
                thetadiff = v @ (self.tX[k, :] - self.tX[m, :])
                    
                result[count, 0] = thetadiff - c * sigma
                result[count, 1] = thetadiff + c * sigma
                    
                count += 1
                
        l = 1
        r = self.n
        for i in range(self.n - 1):
            if result[i, 0] > 0 and result[i, 1] > 0:
                l += 1
            if result[i, 0] < 0 and result[i, 1] < 0:
                r -= 1
                    
        return l, r, c
    
    def GMB1(self, v, num, m = 0):
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
            
        sigma_recorded = np.zeros(self.n)
        up_all_recorded = np.zeros([self.n, self.n])
        scores_recorded = self.tX @ v
        for k in range(self.n):
            if k != m:
                s = 0
                diff = self.tX[m, :] - self.tX[k, :]
                tmp = diamond @ diff
                #sigma_recorded[k] = np.sqrt(tmp.T @ H @ tmp / self.L)
                sigma_recorded[k] = np.sqrt(tmp.T @ H @ tmp)
                up_all_recorded[k, :] = tmp @ self.tX.T
        results = []
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    tmp = scores_recorded[i] - scores_recorded[j]
                    tmp = np.exp(tmp) / (1 + np.exp(tmp))
                    #hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i] * self.L) 
                    hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i]) 
                    #hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j] * self.L)
                    hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j])
                    #hada[i, j] /= self.L
                    if self.graph[i, j] > 0:
                        hada[i, j] /= self.graph[i, j]
            for k in range(self.n):
                if k != m:
                    s = 0
                    for i in range(self.n):
                        for j in range(i):
                            up = up_all_recorded[k, i] - up_all_recorded[k, j]
                            s += up * hada[i, j] / sigma_recorded[k]
                    if np.abs(s) > result:
                        result = np.abs(s)
            results.append(result)
            
        return results
 
    def CI1(self, v, m = 0):
        tmp = self.GMB1(v, 200, m)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
        g = self.gradient(v)
        result = np.zeros([self.n - 1, 2])
        count = 0
        T = 0
        for k in range(self.n):
            if k != m:
                diff = self.tX[m, :] - self.tX[k, :]
                tmp = diamond @ diff
                #sigma = np.sqrt(tmp.T @ H @ tmp / self.L)
                sigma = np.sqrt(tmp.T @ H @ tmp)
                
                thetadiff = (v[k] - (g[k] / H[k, k])) - (v[m] - (g[m] / H[m, m]))
                thetadiff += v[self.n:] @ (self.X[k, :] - self.X[m, :])
                result[count, 0] = thetadiff - c * sigma
                result[count, 1] = thetadiff + c * sigma
                    
                count += 1
                
        l = 1
        r = self.n
        for i in range(self.n - 1):
            if result[i, 0] > 0 and result[i, 1] > 0:
                l += 1
            if result[i, 0] < 0 and result[i, 1] < 0:
                r -= 1
                    
        return l, r, c
    
    def GMB_test(self, v, num, tZ, sigma_recorded, up_all_recorded):
        
        tmp_recorded = np.zeros([14 * 27, 200])
        
        scores_recorded = self.tX @ v
        results = []
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    if self.graph[i, j] > 0:
                        tmp = scores_recorded[i] - scores_recorded[j]
                        tmp = np.exp(tmp) / (1 + np.exp(tmp))
                        hada[i, j] = (tmp - 1) * np.random.randn() * np.sqrt(self.y[j, i]) 
                        hada[i, j] += tmp * np.random.randn() * np.sqrt(self.y[i, j])
                        hada[i, j] /= self.graph[i, j]
            
            tmp_count = 0
            
            for m in range(self.b):
                for k in range(m):
                    s = 0
                    for i in range(self.n):
                        for j in range(i):
                            up = up_all_recorded[m, i] + up_all_recorded[k, j] - up_all_recorded[m, j] - up_all_recorded[k, i]
                            s += up * hada[i, j] / sigma_recorded[m, k]
                    
                    tmp_recorded[tmp_count, it] = s
                    tmp_count += 1

                    
                    if np.abs(s) > result:
                        result = np.abs(s)
            results.append(result)
            np.save('test_Apr11', results)
            
            np.save('Apr16normaltest.npy', tmp_recorded)
            
        return results
    
    def CI_test(self, v, X_test):
        self.b = X_test.shape[0]
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
        
        tZ = np.zeros([self.b, self.n + self.d])
        for i in range(self.b):
            prev_ind_i = indexdict_train[evmap[indexlist_test[i]]]
            tZ[i, self.n:] = self.X[prev_ind_i, :]
            if prev_ind_i in self.estimatedsupport:
                tZ[i, prev_ind_i] = 1
                
        sigma_recorded = np.zeros([self.b, self.b])
        up_all_recorded = np.zeros([self.b, self.n])
        for m in range(self.b):
            for k in range(m):
                diff = tZ[m, :] - tZ[k, :]
                tmp = diamond @ diff
                sigma_recorded[k, m] = np.sqrt(tmp.T @ H @ tmp)
                sigma_recorded[m, k] = np.sqrt(tmp.T @ H @ tmp)
            up_all_recorded[m, :] = (diamond @ tZ[m, :]) @ self.tX.T
            
        print(sigma_recorded)
        
        tmp = self.GMB_test(v, 200, tZ, sigma_recorded, up_all_recorded)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        result = np.zeros([self.b, self.b, 2])
        for m in range(self.b):
            prev_ind_m = indexdict_train[evmap[indexlist_test[m]]]
            for k in range(m):
                prev_ind_k = indexdict_train[evmap[indexlist_test[k]]]
                thetadiff = (X_test[m, :] - X_test[k, :]) @ v[self.n:]
                if prev_ind_m in self.estimatedsupport:
                    thetadiff += v[prev_ind_m]
                if prev_ind_k in self.estimatedsupport:
                    thetadiff -= v[prev_ind_k]

                result[m, k, 0] = thetadiff - c * sigma_recorded[k, m]
                result[m, k, 1] = thetadiff + c * sigma_recorded[k, m]
                
                result[k, m, 0] = -thetadiff - c * sigma_recorded[k, m]
                result[k, m, 1] = -thetadiff + c * sigma_recorded[k, m]
            
        rank = np.zeros([self.b, 2])
        for m in range(self.b):
            l = 1
            r = self.b
            for k in range(self.b):
                if k != m:
                    if result[m, k, 0] > 0:
                        l += 1
                    if result[m, k, 1] < 0:
                        r -= 1
            rank[m, 0] = l
            rank[m, 1] = r
                    
        return rank, c
    
    def GMB_GOF(self, v, num):
        H = self.Hessian(v)
        scores_recorded = self.tX @ v
        results = []
        
        for it in range(num):
            print(it)
            result = -1
            hada = np.zeros([self.n, self.n])
            for i in range(self.n):
                for j in range(i):
                    tmp = scores_recorded[i] - scores_recorded[j]
                    tmp = np.exp(tmp) / (1 + np.exp(tmp))
                    g1 = np.random.randn()
                    g2 = np.random.randn()
                    hada[i, j] = (tmp - 1) * g1 * np.sqrt(self.y[j, i]) 
                    hada[i, j] += tmp * g2 * np.sqrt(self.y[i, j])
                    #hada[i, j] /= np.sqrt(self.L)
                    
                    hada[j, i] = - hada[i, j]
            
            for i in range(self.n):
                s = np.sum(hada[i, :]) / np.sqrt(H[i, i])
                if np.abs(s) > result:
                    result = np.abs(s)

            results.append(result)

        return results
    
    def GoFTest(self, v):
        tmp = self.GMB_GOF(v, 200)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        H = self.Hessian(v)
        g = self.gradient(v)
        T = 0
        for i in range(self.n):
            debiased = v[i] - (g[i] / H[i, i])
            tmp = np.sqrt(H[i, i]) * np.abs(debiased)
            if tmp > T:
                T = tmp
                    
        return T, c
    
    def CI_Stage1(self, v, X_test):
        self.b = X_test.shape[0]
        diamond = self.diamondinv(v)
        H = self.Hessian(v)
        g = self.gradient(v)
        
        tZ = np.zeros([self.b, self.n + self.d])
        for i in range(self.b):
            prev_ind_i = indexdict_train[evmap[indexlist_test[i]]]
            tZ[i, self.n:] = self.X[prev_ind_i, :]
            tZ[i, prev_ind_i] = 1
                
        sigma_recorded = np.zeros([self.b, self.b])
        up_all_recorded = np.zeros([self.b, self.n])
        for m in range(self.b):
            for k in range(m):
                diff = tZ[m, :] - tZ[k, :]
                tmp = diamond @ diff
                sigma_recorded[k, m] = np.sqrt(tmp.T @ H @ tmp)
                sigma_recorded[m, k] = np.sqrt(tmp.T @ H @ tmp)
            up_all_recorded[m, :] = (diamond @ tZ[m, :]) @ self.tX.T
            
        print(sigma_recorded)
        
        tmp = self.GMB_test(v, 200, tZ, sigma_recorded, up_all_recorded)
        tmp.sort()
        c = (tmp[-10] + tmp[-11]) / 2
        
        result = np.zeros([self.b, self.b, 2])
        for m in range(self.b):
            prev_ind_m = indexdict_train[evmap[indexlist_test[m]]]
            for k in range(m):
                prev_ind_k = indexdict_train[evmap[indexlist_test[k]]]
                thetadiff = (X_test[m, :] - X_test[k, :]) @ v[self.n:]
                thetadiff += (v[prev_ind_m] - g[prev_ind_m] / H[prev_ind_m, prev_ind_m])
                thetadiff -= (v[prev_ind_k] - g[prev_ind_k] / H[prev_ind_k, prev_ind_k])

                result[m, k, 0] = thetadiff - c * sigma_recorded[k, m]
                result[m, k, 1] = thetadiff + c * sigma_recorded[k, m]
                
                result[k, m, 0] = -thetadiff - c * sigma_recorded[k, m]
                result[k, m, 1] = -thetadiff + c * sigma_recorded[k, m]
            
        rank = np.zeros([self.b, 2])
        for m in range(self.b):
            l = 1
            r = self.b
            for k in range(self.b):
                if k != m:
                    if result[m, k, 0] > 0:
                        l += 1
                    if result[m, k, 1] < 0:
                        r -= 1
            rank[m, 0] = l
            rank[m, 1] = r
                    
        return rank, c

In [15]:
# prepare data for out-of-sample rank confidence interval

def generate_test_data(test_combats, pokemon_bio, indexlist_train, indexlist_test, means, stds, indexdict_train, indexdict_test):
    n = len(indexlist_train)
    m = len(indexlist_test)
    
    indexset_test = set(indexlist_test)
    
    jointlist = indexlist_train + indexlist_test
    jointdict = {}
    for i in range(len(jointlist)):
        jointdict[jointlist[i]] = i
    
    X_test = np.zeros([m, 3])
    for i in range(len(indexlist_test)):
        X_test[i, 0] = np.log(pokemon_bio.iloc[indexlist_train[i] - 1]["HP"])
        X_test[i, 1] = np.log(pokemon_bio.iloc[indexlist_train[i] - 1]["Attack"])
        X_test[i, 2] = int(indexlist_test[i] in mega)
    
    for j in range(3):
        X_test[:, j] = (2/27) * (X_test[:, j] - means[j]) / stds[j]
    
    y_test = np.zeros([n + m, m])
    graph_test = np.zeros([n + m, m])
    for i in range(len(test_combats)):
        if test_combats.iloc[i]['First_pokemon'] in indexset_test:
            if test_combats.iloc[i]['Second_pokemon'] in indexlist_train:
                graph_test[jointdict[test_combats.iloc[i]['Second_pokemon']], indexdict_test[test_combats.iloc[i]['First_pokemon']]] += 1
                if train_combats.iloc[i]['First_pokemon'] == train_combats.iloc[i]['Winner']:
                    y_test[jointdict[test_combats.iloc[i]['Second_pokemon']], indexdict_test[test_combats.iloc[i]['First_pokemon']]] += 1
            elif test_combats.iloc[i]['Second_pokemon'] in indexlist_test:
                graph_test[jointdict[test_combats.iloc[i]['Second_pokemon']], indexdict_test[test_combats.iloc[i]['First_pokemon']]] += 1
                graph_test[jointdict[test_combats.iloc[i]['First_pokemon']], indexdict_test[test_combats.iloc[i]['Second_pokemon']]] += 1
                if train_combats.iloc[i]['First_pokemon'] == train_combats.iloc[i]['Winner']:
                    y_test[jointdict[test_combats.iloc[i]['Second_pokemon']], indexdict_test[test_combats.iloc[i]['First_pokemon']]] += 1
                else:
                    y_test[jointdict[test_combats.iloc[i]['First_pokemon']], indexdict_test[test_combats.iloc[i]['Second_pokemon']]] += 1
        
        if test_combats.iloc[i]['Second_pokemon'] in indexset_test:
            if test_combats.iloc[i]['First_pokemon'] in indexlist_train:
                graph_test[jointdict[test_combats.iloc[i]['First_pokemon']], indexdict_test[test_combats.iloc[i]['Second_pokemon']]] += 1
                if train_combats.iloc[i]['Second_pokemon'] == train_combats.iloc[i]['Winner']:
                    y_test[jointdict[test_combats.iloc[i]['First_pokemon']], indexdict_test[test_combats.iloc[i]['Second_pokemon']]] += 1
        
    return X_test, y_test, graph_test

def soft(x, thres):
    return np.sign(x) * max(0, abs(x) - 3.99008292685308 * thres)

def loss(y_test, graph_test, X_test, X, tb):
    m = y_test.shape[1]
    n = X.shape[0]
    
    var_total = data.variance(tb)
    diag = np.diag(var_total)
    
    sum_new = 0
    sum_old = 0
    logistic_new = 0
    logistic_old = 0
    for i in range(n):
        for j in range(m):
            if graph_test[i, j] > 0 and evmap[indexlist_test[j]] in indexdict_train.keys():
                prev_ind_j = indexdict_train[evmap[indexlist_test[j]]]
                score_old = tb[prev_ind_j] + X[prev_ind_j, :] @ tb[n:] - tb[i] - X[i, :] @ tb[n:]
                score_new = soft(tb[prev_ind_j], np.sqrt(diag[prev_ind_j])) + X_test[j, :] @ tb[n:] - soft(tb[i], np.sqrt(diag[i])) - X[i, :] @ tb[n:]
                sum_old += ((np.exp(score_old) / (1 + np.exp(score_old))) - y_test[i, j] / graph_test[i, j]) ** 2
                sum_new += ((np.exp(score_new) / (1 + np.exp(score_new))) - y_test[i, j] / graph_test[i, j]) ** 2
                y_tmp = y_test[i, j] / graph_test[i, j]
                logistic_old += np.log(1 + np.exp(score_old)) - y_tmp * score_old
                logistic_new += np.log(1 + np.exp(score_new)) - y_tmp * score_new
                
    
    for i in range(m):
        for j in range(i):
            if graph_test[i + n, j] > 0 and evmap[indexlist_test[j]] in indexdict_train.keys() and evmap[indexlist_test[i]] in indexdict_train.keys():
                prev_ind_i = indexdict_train[evmap[indexlist_test[i]]]
                prev_ind_j = indexdict_train[evmap[indexlist_test[j]]]
                score_old = tb[prev_ind_j] + X[prev_ind_j, :] @ tb[n:] - tb[prev_ind_i] - X[prev_ind_i, :] @ tb[n:]
                score_new = soft(tb[prev_ind_j], np.sqrt(diag[prev_ind_j])) + X_test[j, :] @ tb[n:] - soft(tb[prev_ind_i], np.sqrt(diag[prev_ind_i])) - X_test[i, :] @ tb[n:]
                sum_old += ((np.exp(score_old) / (1 + np.exp(score_old))) - y_test[i + n, j] / graph_test[i + n, j]) ** 2
                sum_new += ((np.exp(score_new) / (1 + np.exp(score_new))) - y_test[i + n, j] / graph_test[i + n, j]) ** 2
                y_tmp = y_test[i + n, j] / graph_test[i + n, j]
                logistic_old += np.log(1 + np.exp(score_old)) - y_tmp * score_old
                logistic_new += np.log(1 + np.exp(score_new)) - y_tmp * score_new
        
    return sum_old, sum_new, logistic_old, logistic_new
                  

# Fit the Data

In [140]:
# prepare test data
X_test, y_test, graph_test = generate_test_data(test_combats, pokemon_bio, indexlist_train, indexlist_test, means, stds, indexdict_train, indexdict_test)

In [ ]:
# fit the model using training data
data = Ranking(X, y, graph)
tx = data.generate_MLE(lamb = 30)

# calculate the rank confidence interval on processed test data (state I)
CI_test_all_stage1 = data.CI_Stage1(tx, X_test)
np.save('test_May31_rank_stage1.npy', CI_test_all_stage1[0])
np.save('test_May31_c_stage1.npy', CI_test_all_stage1[1])

0 0.741839451289551
1 0.8028865038724442
2 0.818003591638854
3 0.7972065749456034
4 0.7506034418993022
5 0.6868662057626665
6 0.6145429617005762
7 0.539520260343979
8 0.4673444504571794
9 0.40233643146004405
10 0.3474381093303424
11 0.30485056010491496
12 0.2715866081765207
13 0.24970515679277377
14 0.23321722468813805
15 0.21978881445243803
16 0.2062137761370078
17 0.19274472342788695
18 0.17757222801920303
19 0.1598947323989686
20 0.14193056308113852
21 0.12420108490065979
22 0.108073637163751
23 0.09366424187283368
24 0.08270488468843291
25 0.07465021109483304
26 0.06888725678898261
27 0.06407368660171295
28 0.0598518110334146
29 0.05567574141528155
30 0.05117964831369126
31 0.046425177215473935
32 0.04164217657611719
33 0.03713325447454771
34 0.033190938730944995
35 0.030028342630138722
36 0.027731913519036856
37 0.026254928717022324
38 0.025459588601401118
39 0.025176263737001752
40 0.025269265824263667
41 0.025618057337172814
42 0.026117567976505694
43 0.026657974693741763
44 0.0

1
2
3


In [ ]:
# refit the model, calculate the rank confidence interval on processed test data (stage II)
data.fix_support(tx)
tx_refit = data.generate_MLE_support()
CI_test_all = data.CI_test(tx_refit, X_test)
np.save('test_May31_rank.npy', CI_test_all[0])
np.save('test_May31_c.npy', CI_test_all[1])

In [ ]:
# goodness-of-fit test
data = Ranking(X, y, graph)
results = np.zeros([5, 3])
penalty = [10,20,30,40,50]

for i in range(5):
    print(i)
    tx = data.generate_MLE(lamb = penalty[i])
    tmp = data.GoFTest(tx)
    results[i, 0] = tmp[0]
    results[i, 1] = tmp[1]
    results[i, 2] = np.sum(tx[:-3]!=0)
    np.save('GOF_May31_1050.npy', results)

In [157]:
# rank cofidence interval of some selected ids
index_selected = [item for item in indexlist_test if item % 5 == 0]
for i in index_selected:
    print(i, CI_test_all_stage1[0][indexdict_test[i], :], CI_test_all[0][indexdict_test[i], :])

125 [ 7. 24.] [ 7. 10.]
225 [1. 6.] [14. 17.]
340 [14. 27.] [ 8. 10.]
230 [1. 6.] [1. 1.]
155 [23. 28.] [23. 25.]
495 [ 7. 24.] [13. 13.]
20 [ 7. 25.] [20. 23.]


In [159]:
# goodness-of-fit test results
results

array([[ 12.3424091 ,   3.75909033, 502.        ],
       [ 10.76495534,   4.20171839, 283.        ],
       [ 10.04148805,   4.29557342,  84.        ],
       [ 10.05087154,   4.20649281,   7.        ],
       [ 10.2060637 ,   4.28329473,   0.        ]])